# Reachability-based BLAs for intersection crossings

Companion to [protocol.ipynb](protocol.ipynb) for Sec. IV, BLAs with spatio-temporal resources.

- Cells tagged `definitions` are the method.
- Kernel: Python 3.13, `pyspect[hj_reachability]==1.2`.
- Evidence: **[algebra]** normalization · **[semantic]** containment and the client's driving limits ·
  **[numerical]** grid membership.
- Pipeline: request $\theta$ → contracts → quotient → TLT of $\varphi_j$ (admission) → pruned corridors (offers)
  → bargaining.
- Scope: point-mass kinematic bicycle on a four-way intersection, sampled entry sets, nominal HJ, a finite list
  of exit windows. No robustness certificate.

In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore', message='IProgress not found')  # tqdm without ipywidgets
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src' / 'ltms_reachability').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from ltms_bla.presentation import table

In [2]:
from dataclasses import dataclass
from fractions import Fraction as F
from math import pi
from time import perf_counter
import interval_logic as il
import portion as P
import numpy as np
from pyspect import Set
from pyspect.set_builder import BoxSet, Inter, PolytopeSet, Union
from pyspect.impls.hj_reachability import TVHJImpl
from pyspect.systems.hj_reachability import Bicycle4D
from protocol import CommitResult, Contract, affine_policies, normalize, validate_commit
from ltms_reachability import LTMSPlanner, NoCorridorError, TimeWindow, WindowPolicy
from ltms_bla.bargaining import load_bargain
from ltms_bla.tlt import compile_query

## 1. Model

The intersection manager (owner) decides the admissible trajectories $\xi^\star_j$; vehicle $j$ (client) controls
$u=(\delta,a)$ and decides which of them it takes.
$$\dot x=v\cos h,\quad \dot y=v\sin h,\quad \dot h=v\tan\delta/L,\quad \dot v=a .$$
Point vehicle at $(x,y)$ with heading $h$ in the global frame.

In [3]:
WHEELBASE = .32                                                                    # L [m]
CONTROLS = dict(min_steer=-pi/4, max_steer=pi/4, min_accel=-.15, max_accel=.15)  # δ [rad], a [m/s²]
SPEED_LIMIT = (.3, .6)                                                             # v [m/s] on every road
VMAX = SPEED_LIMIT[1]

**Map.** A four-way intersection with right-hand traffic, as in the original LTMS.
- Each direction of travel has a lane, entered at one end and left at the other.
- On its lane a vehicle keeps within $\pm\pi/5$ of the lane's heading; in the centre (an octagon) any heading.
- $\mathit{road}_d$ = lane $d$ with its heading band, or the centre. $\mathit{exit}_d$ = exit zone of lane $d$.
- A route enters travelling *inbound* and leaves travelling *outbound* (straight on, a turn or a U-turn).
- The intersection region $\mathcal X$ is the whole map.

In [4]:
# Direction of travel → heading [rad], lane, entry zone and exit zone (boxes over x, y [m]).
MAP = dict(
    east=dict(heading=0, lane=dict(y=(-.8, 0)),
              entry=dict(x=(-2, -1.4), y=(-.68, -.12)), exit=dict(x=(1.4, 2), y=(-.68, -.12))),
    north=dict(heading=pi/2, lane=dict(x=(0, .8)),
               entry=dict(x=(.12, .68), y=(-2, -1.4)), exit=dict(x=(.12, .68), y=(1.4, 2))),
    west=dict(heading=pi, lane=dict(y=(0, .8)),
              entry=dict(x=(1.4, 2), y=(.12, .68)), exit=dict(x=(-2, -1.4), y=(.12, .68))),
    south=dict(heading=-pi/2, lane=dict(x=(-.8, 0)),
               entry=dict(x=(-.68, -.12), y=(1.4, 2)), exit=dict(x=(-.68, -.12), y=(-2, -1.4))),
)
HEADING_BAND = pi/5  # on its lane, a vehicle keeps within ±π/5 of the lane's heading
MANAGED = 1.2        # the centre: the octagon |x| + |y| ≤ 2 within [-1.2, 1.2]²

**Request and resource rules.** Request $\theta=(J,E_{\rm in},E_{\rm out})$: enter through $E_{\rm in}$ during $J$,
leave through $E_{\rm out}=\mathit{exit}_{\rm out}$.
$$\varphi_j:=(\mathcal C_j\land\neg\mathcal D_j)\ \mathcal U\ E_{\rm out},\qquad
\mathcal C_j:=\mathit{road}_{\rm in}\lor\mathit{road}_{\rm out},$$
$$R:=\square(\mathcal X\to\Phi_j),\qquad S_u:=\lozenge_J E_{\rm in},\qquad
S_a:=\lozenge_J\big(E^\star_{\rm in}\land(\mathcal C^\star_j\ \mathcal U\ E^\star_{\rm out})\big),\qquad
C_{\rm req}=\langle S_a\mid R\land S_u\rangle .$$
- Access: being inside one's corridor $\Phi_j$. Use: being inside $\mathcal X$.
- $R$, $S_u$ constrain the vehicle's $\xi_j$; $S_a$ (starred atoms) constrains every admissible $\xi^\star_j$.
- $\mathcal D_j$: time-states that risk collision with standing corridors (§2).

In [5]:
# (x, y, h, v) at entry: the middle of each entry zone, heading along the lane.
ENTRY = dict(east=(-1.6, -.4, 0, .45), north=(.4, -1.6, pi/2, .45),
             west=(1.6, .4, pi, .45), south=(-.4, 1.6, -pi/2, .45))


@dataclass(frozen=True)
class Crossing:
    # θ = (J, E_in, E_out): enter travelling inbound during J = [a, b] [s], leave travelling outbound.
    client: str
    inbound: str
    outbound: str
    entry: tuple

    def atom(self, name, admissible=None):
        # None: a map atom, interpreted in §2. False: read on vehicle j's ξ_j. True: on an admissible ξ*_j.
        return il.Proposition(name if admissible is None else f'{name}_{self.client}' + '*'*admissible)

    def road(self, admissible=None):
        # C_j: the inbound road, and the outbound road when turning.
        inbound, outbound = (self.atom(f'road_{d[0]}', admissible) for d in (self.inbound, self.outbound))
        return inbound if self.inbound == self.outbound else inbound | outbound

    def entering(self, admissible=False):
        return self.atom(f'in_{self.inbound[0]}', admissible)

    def exiting(self, admissible=None):
        return self.atom(f'exit_{self.outbound[0]}', admissible)

    @property
    def corridor(self):
        return il.Proposition(f'Phi_{self.client}')

    @property
    def rule(self):  # R
        return il.Always(il.Implies(self.atom('region', False), self.corridor))

    @property
    def use(self):  # S_u
        return il.Eventually(self.entering(False), P.closed(*self.entry))

    @property
    def access(self):  # S_a
        crossing = il.Until(self.road(True), self.exiting(True))
        return il.Eventually(self.entering(True) & crossing, P.closed(*self.entry))

    def query(self, danger):
        # φ_j with D_j the atom `danger`, over the grid's horizon from J's start.
        return il.Until(self.road() & ~danger, self.exiting(), P.closed(self.entry[0], self.entry[0] + HORIZON))

| | Meaning | Constrains | Actor |
|:--|:--|:--|:--|
| $S_u$ | enter through $E_{\rm in}$ during $J$ | $\xi_j$ | client |
| $R$ | inside $\mathcal X$ only inside the corridor $\Phi_j$ | $\xi_j$ | client |
| $S_a$ | every admissible trajectory enters through $E_{\rm in}$ during $J$ and keeps to $\mathcal C_j$ until $E_{\rm out}$ | $\xi^\star_j$ | owner |
| $H_\kappa$ | the admissible trajectories are those of the offered corridor $\Phi_\kappa$ | $\xi^\star_j$ | owner (offer) |

**[semantic]** $R$ is the time-state form of following an admissible trajectory inside $\mathcal X$. The vehicle
realizes $C_{\rm req}$ by following the driving limits of $\Phi_j$; under $R$ it leaves through $E_{\rm out}$ before
the corridor ends.

## 2. Environment, query and corridors

Grid over $(t,x,y,h,v)$: 10 s in 0.25 s steps. The heading axis is periodic and covers one turn from $-3\pi/4$,
so no lane's heading band wraps around. §1's map becomes pyspect set builders (inside ⇔ $\le0$).
A request with $J=[a,b]$ is solved on $[a,a+10]$.

In [6]:
HORIZON, DT = 10, .25
AXES = [dict(name='t', bounds=[0, HORIZON], step=DT, unit='s'),
        dict(name='x', bounds=[-2, 2], points=41, unit='m'), dict(name='y', bounds=[-2, 2], points=41, unit='m'),
        dict(name='*h', bounds=[-3*pi/4, 5*pi/4], points=16, unit='rad'),  # '*': periodic
        dict(name='v', bounds=list(SPEED_LIMIT), points=5, unit='m/s')]
impl = TVHJImpl(dict(cls=Bicycle4D, wheelbase=WHEELBASE, **CONTROLS), AXES, accuracy='medium')
x, y, h, v = (np.asarray(c) for c in impl.grid.coordinate_vectors)


def box(**bounds):
    return BoxSet(bounds, axes=list(bounds))


OCTAGON = PolytopeSet(normals=[[1, 1], [-1, 1], [-1, -1], [1, -1]],  # |x| + |y| ≤ 2
                      offsets=[[-1, -1], [1, -1], [1, 1], [-1, 1]], axes=['x', 'y'])
SPEED = box(v=SPEED_LIMIT)
CENTRE = Inter(Inter(OCTAGON, box(x=(-MANAGED, MANAGED), y=(-MANAGED, MANAGED))), SPEED)  # any heading
ROAD = {d: Union(Inter(box(**m['lane'], h=(m['heading'] - HEADING_BAND, m['heading'] + HEADING_BAND)), SPEED),
                 CENTRE) for d, m in MAP.items()}
ENTRY_ZONE = {d: Inter(box(**m['entry']), ROAD[d]) for d, m in MAP.items()}
EXIT_ZONE = {d: Inter(box(**m['exit']), ROAD[d]) for d, m in MAP.items()}
INTERPRETATION = {f'road_{d[0]}': ROAD[d] for d in MAP} | {f'exit_{d[0]}': EXIT_ZONE[d] for d in MAP}

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [7]:
# Roads (grey), entry zones (blue) and exit zones (red), projected over (t, h, v); arrows at the entry states.
figure = impl.plot(*((ROAD[d](impl), dict(colorscale='Greys', value=.3)) for d in MAP),
                   *((ENTRY_ZONE[d](impl), dict(colorscale='Blues', value=.6)) for d in MAP),
                   *((EXIT_ZONE[d](impl), dict(colorscale='Reds', value=.6)) for d in MAP),
                   method='bitmap', axes=('x', 'y'),
                   layout=dict(title='Four-way intersection', width=560, height=540))
for d, (px, py, ph, _) in ENTRY.items():
    figure.add_annotation(x=px + .5*np.cos(ph), y=py + .5*np.sin(ph), ax=px, ay=py, axref='x', ayref='y',
                          text=d, arrowhead=2, arrowwidth=2)
figure.show()

**Entry sets, lanes and danger.**
- $E_{\rm in}$: a box around the entry state ($\pm0.1$ m, $\pm\pi/8$, $\pm0.08$ m/s), inside the entry zone.
- Lanes: an offer reserves only the middle strip of each lane it uses and, for a turn, the lanes' common square;
  a U-turn may turn anywhere across both lanes in the centre.
- $\mathcal D_j$: the standing corridors' cells on the grid's absolute times, dilated by both bodies
  ($2\times0.075$ m).

In [8]:
STRIP_WIDTH, BODY = .4, .075  # [m]; BODY: radius of the vehicle around its point
E_IN = {d: box(x=(px - .1, px + .1), y=(py - .1, py + .1), h=(ph - pi/8, ph + pi/8), v=(pv - .08, pv + .08))
        for d, (px, py, ph, pv) in ENTRY.items()}


def strip(d):
    # The middle of lane d, along the whole map.
    (axis, (lo, hi)), = MAP[d]['lane'].items()
    return box(**{axis: ((lo + hi - STRIP_WIDTH)/2, (lo + hi + STRIP_WIDTH)/2)})


def lanes(service):
    # The part of the feasible set an offer reserves.
    lane_in, lane_out = MAP[service.inbound]['lane'], MAP[service.outbound]['lane']
    if service.inbound == service.outbound:
        return strip(service.inbound)
    if lane_in.keys() == lane_out.keys():  # U-turn: across both lanes, inside the centre
        (axis, a), = lane_in.items()
        (_, b), = lane_out.items()
        corner = box(**{axis: (min(a[0], b[0]), max(a[1], b[1])), 'y' if axis == 'x' else 'x': (-MANAGED, MANAGED)})
    else:
        corner = box(**lane_in, **lane_out)
    return Union(Union(strip(service.inbound), strip(service.outbound)), corner)


def danger(ledger, origin):
    # D_j on the grid's times [origin, origin + HORIZON]: every standing corridor's cells, dilated by both bodies.
    times = np.asarray(impl.timeline) + float(origin)
    cells = np.zeros((len(times), len(x), len(y)), bool)
    for bla in ledger:
        cells |= bla.offer.reserved.on_grid(times, x, y, ego_radius=BODY)
    return Set(np.broadcast_to(np.where(cells, -1., 1.)[..., None, None], impl.shape))

**Query and corridor [numerical].** `compile_query` turns $\varphi_j$, anchored at $a$, into a TLT whose root is
$\mathrm{sat}^\exists(\varphi_j)$. An offer prunes it with the LTMS passes for an exit window $K$:
- pass 3: forward from the entry zone during $J$, inside $\mathrm{sat}^\exists(\varphi_j)$ and the lanes;
- pass 4: backward from $E_{\rm out}$ during $K$, inside pass 3. This is the corridor $\Phi_\kappa$.

Membership is read at the grid points of $E_{\rm in}\times J$ (inside if $\le 2\cdot10^{-5}$).

In [9]:
TOLERANCE = 2e-5
PLANNER = LTMSPlanner(impl)


def crossing_set(service, ledger):
    # sat∃(φ_j): the root of the TLT generated from φ_j, with D_j from the standing corridors.
    D = il.Proposition(f'D_{service.client}')
    tree = compile_query(service.query(D), service.entry[0],
                         INTERPRETATION | {D.name: danger(ledger, service.entry[0])})
    return np.asarray(tree.realize(impl))


def covers(values, service):
    # E_in × J ⊆ {values ≤ 0}, with J in the grid's local time.
    a, b = service.entry
    during = np.asarray(impl.timeline) <= b - a + 1e-6
    inside = np.broadcast_to(np.asarray(E_IN[service.inbound](impl)), impl.shape)[during] <= 0
    return bool(np.all(values[during][inside] <= TOLERANCE))


def corridor(service, crossing_set, exit):
    # Passes 3-4 for local exit window `exit` [s]; passes 1-2 are sat∃(φ_j) within the lanes.
    duration = service.entry[1] - service.entry[0]
    return PLANNER.analyze(entry=ENTRY_ZONE[service.inbound], goal=EXIT_ZONE[service.outbound],
                           constraints=crossing_set, offline=np.maximum(crossing_set, lanes(service)(impl)),
                           entry_window=TimeWindow(0, duration), exit_window=TimeWindow(*exit),
                           policy=WindowPolicy(duration, duration, 0, exit[1] - exit[0]),
                           time_origin=float(service.entry[0]))

## 3. Contracts, admission and offers

$$C_{\rm trg}=\langle\top\mid R\land S_u\rangle,\quad Q=C_{\rm trg}/C_{\rm req}=\langle\widehat G_{\rm req}\mid S_a\rangle,\quad
C_{\rm cmt}^{+}=C_{\rm cmt}\wedge Q=\langle\top\mid L\land S_a\rangle .$$
- Access-only again: $S_a$ only restricts which trajectories the owner admits.
- **Entry condition** $E_{\rm in}\times J\subseteq\mathrm{sat}^\exists(\varphi_j)$: the vehicle may enter anywhere in
  $E_{\rm in}\times J$, so the corridor must admit a crossing from each.
- Try the original service, then permitted alternatives **in order** (another $J$ is an adaptation). A service is
  selected only if it passes the entry condition **and** offer generation (§III-C) yields a certified offer;
  the entry condition reads $\mathrm{sat}^\exists(\varphi_j)$, while offers keep to the lanes, so it is necessary only.
  Exhaustion returns `None` (`NO-BLA`).

In [10]:
@dataclass(frozen=True)
class Request:
    client: str
    service: Crossing
    alternatives: tuple = ()

    @property
    def services(self):
        # The original service, then permitted adaptations in order.
        return (self.service, *self.alternatives)


@dataclass(frozen=True, eq=False)
class Admission:
    request: Request
    service: Crossing
    client_contract: Contract  # C_req
    task: Contract             # C_trg
    quotient: Contract         # Q
    obligations: Contract      # C_cmt^+
    commitment: Contract       # C_cmt, frozen for this attempt
    crossing_set: object       # sat∃(φ_j) on the grid from J's start
    candidates: tuple          # its candidate offers (§3, Offers)


def service_contracts(service):
    # IV-B: the client contract and the joint target for one selected service.
    G = service.rule & service.use
    return Contract(service.access, G), Contract(True, G)


def standing(ledger):
    # C_cmt = <T | L>, the conjunction of every accepted BLA.
    result = Contract(True, True)
    for bla in ledger:
        result = result.conjoin(bla.contract)
    return result


def admit(request, ledger, tried=None):
    # Frozen snapshot: the standing commitments stay fixed for this attempt. `tried` logs each service's fate.
    tried = [] if tried is None else tried
    ledger = tuple(ledger)
    commitment = standing(ledger)

    # III-B, Service selection: original service, then permitted alternatives.
    for service in request.services:
        C_req, C_trg = service_contracts(service)

        # IV-C: the quotient again leaves an access-only obligation.
        Q = C_trg.quotient(C_req).normalize()
        assert Q.guarantee == normalize(service.access)
        C_plus = commitment.conjoin(Q)

        # IV-C: the entry condition against the standing corridors.
        crossing = crossing_set(service, ledger)
        if not covers(crossing, service):
            tried.append((service.entry, 'entry condition fails'))
            continue

        # III-B: select the service only if it admits a certified, compatible offer.
        pool = candidates(service, crossing)
        if not generate(pool):
            tried.append((service.entry, 'no certified offer'))
            continue
        tried.append((service.entry, 'admitted'))
        return Admission(request, service, C_req, C_trg, Q, C_plus, commitment, crossing, pool)
    return None

**Offers.** For each candidate exit window $K$: the corridor $\Phi_\kappa$ and $K_\kappa=\langle\top\mid H_\kappa\land S_a\rangle$.
- Exit windows scale with the route: $K=T\cdot[s,s+0.2]$ for $s\in\{0.75,0.9,1.05\}$, with $T$ the nominal travel
  time, the route's centreline length (turns on arcs of radius 0.4 m) at the entry speed. Every route then asks for
  the same relative speeds; an earlier window keeps only faster trajectories, so its corridor is thinner in time.
- **Certified** when $E_{\rm in}\times J\subseteq\Phi_\kappa$: the corridor covers every requested entry, and its
  trajectories enter during $J$ and keep to $\mathcal C_j$ until $E_{\rm out}$. So $H_\kappa\Rightarrow S_a$
  **[semantic]**, recorded by the conjunct $S_a$. Only certified candidates are offered, as in (7).
- **Compatible** by construction: $\Phi_\kappa\subseteq\mathrm{sat}^\exists(\varphi_j)$ avoids $\mathcal D_j$.
- **[algebra]** $K_\kappa\preceq Q$, $C_{\rm cmt}\wedge K_\kappa\preceq C_{\rm cmt}^{+}$,
  $C_{\rm req}\parallel(C_{\rm cmt}\wedge K_\kappa)\preceq C_{\rm cmt}\wedge C_{\rm trg}$.

In [11]:
# Centreline length [m] from the entry state to the exit zone; turns on arcs of radius 0.4 m.
LENGTH = {'straight': 3.0, 'left': 3.6, 'right': 2.0, 'U-turn': 1.9}
EXIT_SCALES = (.75, .9, 1.05)  # candidate K = T·[s, s + 0.2], after J's start [s]


def manoeuvre(service):
    quarter_turns = round((MAP[service.outbound]['heading'] - MAP[service.inbound]['heading']) / (pi/2)) % 4
    return ('straight', 'left', 'U-turn', 'right')[quarter_turns]


def exit_windows(service):
    # T: nominal travel time at the entry speed, on the time grid.
    T = LENGTH[manoeuvre(service)] / ENTRY[service.inbound][3]
    return tuple((round(T*s/DT)*DT, round(T*(s + .2)/DT)*DT) for s in EXIT_SCALES)


@dataclass(frozen=True, eq=False)
class Offer:
    service: Crossing
    index: int
    exit: tuple       # K, absolute [s]
    corridor: object  # Φ_κ, pass 4 on the grid from J's start
    reserved: object  # its cells on absolute time (SpatialReservation)
    certified: bool   # E_in × J ⊆ Φ_κ

    @property
    def start(self):  # proposals try earlier exit windows first
        return self.exit[0]

    @property
    def promise(self):  # H_κ
        return il.Proposition(f'H_{self.service.client}_{self.index}')

    @property
    def contract(self):
        return Contract(True, self.promise & self.service.access)


def candidates(service, crossing_set):
    # IV-C: one corridor per exit window that the passes can realize.
    result = []
    for k, exit in enumerate(exit_windows(service), 1):
        try:
            analysis = corridor(service, crossing_set, exit)
        except NoCorridorError:
            continue
        K = analysis.exit_window
        result.append(Offer(service, k, (analysis.time_origin + K.start, analysis.time_origin + K.end),
                            analysis.corridor, analysis.reservation(body_radius=BODY), covers(analysis.corridor, service)))
    return tuple(result)


def generate(candidates):
    # (7): only certified candidates are offered.
    return tuple(o for o in candidates if o.certified)

**Bargaining and commitment.** `bargain` is verbatim from `protocol.ipynb`; its validator accepts only certified
offers of one selected service. Commit stores the exact accepted pair, $C_{\rm cmt}\leftarrow C_{\rm cmt}\wedge K^*$,
and the accepted corridor enters $\mathcal D$ of later requests. `NO-BLA` preserves commitments and ledger.

In [12]:
def validate_offers(offers):
    if any(not isinstance(o, Offer) or not o.certified for o in offers):
        raise ValueError('bargaining accepts only certified offers')
    if len({o.service for o in offers}) > 1:
        raise ValueError('offers must share one selected service')


bargain = load_bargain(ROOT, validate_offers)


@dataclass(frozen=True)
class BLA:
    offer: Offer
    price: F

    @property
    def contract(self):
        return self.offer.contract


def commit(admission, offers, negotiation, commitment, ledger=()):
    ledger = tuple(ledger)
    if negotiation.status == 'NO-BLA' and negotiation.agreement is None:
        return CommitResult('NO-BLA', commitment, ledger, None)
    # Check the frozen commitment/offers and exact accepted pair; do not regenerate.
    validate_commit(admission, offers, negotiation, commitment)
    record = BLA(negotiation.agreement.offer, negotiation.agreement.price)
    return CommitResult('COMMITTED', commitment.conjoin(record.contract), (*ledger, record), record)


def negotiate(spec, offers):
    # The client's utilities per exit window, earliest first.
    policies = affine_policies({o: spec['costs'][o.index - 1] for o in offers},
                               {o: spec['values'][o.index - 1] for o in offers})
    return bargain(offers, policies, spec['targets'], spec['minima'], PRICES, ROUND_BUDGET)


def attempt(spec, ledger):
    # admit → generate → bargain → commit for one request against the frozen ledger; timed, compute included.
    started = perf_counter()
    ledger = tuple(ledger)
    tried = []
    admission = admit(spec['request'], ledger, tried)
    offers = generate(() if admission is None else admission.candidates)
    negotiation = negotiate(spec, offers)
    result = commit(admission, offers, negotiation, standing(ledger), ledger)
    return dict(request=spec['request'], admission=admission, offers=offers, negotiation=negotiation,
                result=result, tried=tuple(tried), elapsed=perf_counter() - started)

## 4. Staged execution

| Client | Route | $J$ [s] | Case |
|:--|:--|:--|:--|
| A | east | [0, 0.5] | first crossing |
| B | west | [0, 0.5] | simultaneous with A, in the opposite lane |
| C | north | [0, 0.5], then [5, 5.5] | crosses A and B: passes the entry condition, but no certified offer fits its lane; permitted later window |
| D | south → west | [5, 5.5] | right turn, simultaneous with C |
| E | east | [6, 6.5] | price failure |
| F | north → east | [8, 8.5] | right turn |
| G | east → west | [8, 8.5] | U-turn, simultaneous with F |
| E | east | [8, 8.5], then [9, 9.5] | E's new request: G's U-turn fails the entry condition of both, so admission is exhausted |
| H | west → south | [10, 10.5] | left turn |
| I | east → north | [10, 10.5] | opposing left turn, simultaneous with H |

Each $J$ is the earliest whole second at which the case holds. Costs/values per exit window, earliest first:
8/12, 5/10, 2/8; E values 2/1/0.

In [13]:
PRICES, ROUND_BUDGET = (F(0), F(20)), 3
PREFERENCES = dict(costs=(F(8), F(5), F(2)), values=(F(12), F(10), F(8)),
                   targets={'owner': F(4), 'client': F(4)}, minima={'owner': F(1), 'client': F(1)})
PRICE_FAILURE = dict(costs=(F(8), F(5), F(2)), values=(F(2), F(1), F(0)),
                     targets={'owner': F(4), 'client': F(1)}, minima={'owner': F(2), 'client': F(0)})


def window(a):
    # J = [a, a + 0.5] [s]
    return (a, a + F(1, 2))


REQUESTS = (
    dict(request=Request('A', Crossing('A', 'east', 'east', window(0))), **PREFERENCES),
    dict(request=Request('B', Crossing('B', 'west', 'west', window(0))), **PREFERENCES),
    dict(request=Request('C', Crossing('C', 'north', 'north', window(0)),
                         alternatives=(Crossing('C', 'north', 'north', window(5)),)), **PREFERENCES),
    dict(request=Request('D', Crossing('D', 'south', 'west', window(5))), **PREFERENCES),
    dict(request=Request('E', Crossing('E', 'east', 'east', window(6))), **PRICE_FAILURE),
    dict(request=Request('F', Crossing('F', 'north', 'east', window(8))), **PREFERENCES),
    dict(request=Request('G', Crossing('G', 'east', 'west', window(8))), **PREFERENCES),
    dict(request=Request('E', Crossing('E', 'east', 'east', window(8)),
                         alternatives=(Crossing('E', 'east', 'east', window(9)),)), **PRICE_FAILURE),
    dict(request=Request('H', Crossing('H', 'west', 'south', window(10))), **PREFERENCES),
    dict(request=Request('I', Crossing('I', 'east', 'north', window(10))), **PREFERENCES),
)

In [14]:
# Display names. Map atoms as in §1; vehicle k's atoms carry k, starred when read on an admissible ξ*_k.
SYMBOLS = {f'{name}_{d[0]}': rf'\mathit{{{name}}}_{d[0]}' for name in ('road', 'exit') for d in MAP}
for spec in REQUESTS:
    k = spec['request'].client
    for star, mark in (('', ''), ('*', r'\star')):
        SYMBOLS[f'region_{k}{star}'] = rf'\mathcal{{X}}^{{{k}{mark}}}'
        for d in MAP:
            SYMBOLS |= {f'in_{d[0]}_{k}{star}': rf'E^{{{k}{mark}}}_{{\rm in}}',
                        f'exit_{d[0]}_{k}{star}': rf'E^{{{k}{mark}}}_{{\rm out}}',
                        f'road_{d[0]}_{k}{star}': rf'\mathit{{road}}^{{{k}{mark}}}_{d[0]}'}
    SYMBOLS |= {f'Phi_{k}': rf'\Phi_{k}', f'D_{k}': rf'\mathcal{{D}}_{k}'}
    SYMBOLS |= {f'H_{k}_{i}': rf'H^{k}_{i}' for i in range(1, len(EXIT_SCALES) + 1)}


def decimal(value):
    # Rationals and floats as .1 decimals, also inside tuples.
    if isinstance(value, tuple):
        return tuple(decimal(v) for v in value)
    return f'{float(value):.1f}' if isinstance(value, (int, float, F)) and not isinstance(value, bool) else value


def refines(first, second):
    # C1 ⪯ C2: A2 ⇒ A1 and G1 ⇒ G2, decided by normalization [algebra].
    implies = lambda p, q: normalize(p & ~q) == il.Boolean(False)
    return implies(second.assumption, first.assumption) and implies(first.guarantee, second.guarantee)

### B: admission and offers against A

In [15]:
attempts = [attempt(REQUESTS[0], ())]  # A
ledger = attempts[-1]['result'].ledger
B_spec = REQUESTS[1]
started = perf_counter()
tried = []
admission = admit(B_spec['request'], ledger, tried)
pool = admission.candidates
offers = generate(pool)
elapsed = perf_counter() - started
service = admission.service
table(['Object [algebra]', 'Value'], [
    ('$S_a$', service.access), ('$R$', service.rule), ('$S_u$', service.use),
    (r'$C_{\rm req}$', admission.client_contract), (r'$C_{\rm trg}$', admission.task), ('$Q$', admission.quotient),
    (r'$C_{\rm cmt}$ (A)', admission.commitment), (r'$C_{\rm cmt}^{+}$', admission.obligations),
    (r'$\varphi_B$', service.query(il.Proposition('D_B'))),
    (r'$E_{\rm in}\times J\subseteq\mathrm{sat}^\exists(\varphi_B)$ [numerical]', covers(admission.crossing_set, service))],
    SYMBOLS)
table(['Candidate', 'Exit window $K$ [s]', 'Reserved cells', r'$E_{\rm in}\times J\subseteq\Phi_\kappa$ [numerical]',
       r'$K_\kappa\preceq Q$', r'$C_{\rm cmt}\wedge K_\kappa\preceq C_{\rm cmt}^{+}$',
       r'$C_{\rm req}\parallel(C_{\rm cmt}\wedge K_\kappa)\preceq C_{\rm cmt}\wedge C_{\rm trg}$'],
      [(f'κ{o.index}', o.exit, int(o.reserved.positions.sum()), o.certified,
        refines(o.contract, admission.quotient),
        refines(admission.commitment.conjoin(o.contract), admission.obligations),
        refines(admission.client_contract.compose(admission.commitment.conjoin(o.contract)),
                admission.commitment.conjoin(admission.task))) for o in pool])
assert [o.index for o in offers] == [2, 3]

Running for 40 iterations:   0%|          | 0/40 [00:00<?, ?it/s]

Running for 40 iterations:  10%|█         | 4/40 [00:00<00:01, 21.61it/s]

Running for 40 iterations:  20%|██        | 8/40 [00:00<00:01, 22.02it/s]

Running for 40 iterations:  30%|███       | 12/40 [00:00<00:01, 22.14it/s]

Running for 40 iterations:  40%|████      | 16/40 [00:00<00:01, 22.28it/s]

Running for 40 iterations:  50%|█████     | 20/40 [00:00<00:00, 22.61it/s]

... (16 outputs left out)

Running for 40 iterations: 100%|██████████| 40/40 [00:01<00:00, 26.01it/s]

Running for 40 iterations: 100%|██████████| 40/40 [00:01<00:00, 24.66it/s]

| Object [algebra] | Value |
| --- | --- |
| $S_a$ | $\lozenge_{[0,\frac{1}{2}]} \left(E^{B\star}_{\rm in} \land \mathit{road}^{B\star}_w \mathbin{\mathrm{U}} E^{B\star}_{\rm out}\right)$ |
| $R$ | $\square \left(\mathcal{X}^{B} \to \Phi_B\right)$ |
| $S_u$ | $\lozenge_{[0,\frac{1}{2}]} E^{B}_{\rm in}$ |
| $C_{\rm req}$ | $\displaystyle \left\langle \lozenge_{[0,\frac{1}{2}]} \left(E^{B\star}_{\rm in} \land \mathit{road}^{B\star}_w \mathbin{\mathrm{U}} E^{B\star}_{\rm out}\right) \;\middle\vert\; \lozenge_{[0,\frac{1}{2}]} E^{B}_{\rm in} \land \square \left(\Phi_B \lor \neg \mathcal{X}^{B}\right) \lor \square_{[0,\frac{1}{2}]} \left(\neg E^{B\star}_{\rm in} \lor \neg \mathit{road}^{B\star}_w \mathbin{\mathrm{R}} \neg E^{B\star}_{\rm out}\right) \right\rangle$ |
| $C_{\rm trg}$ | $\displaystyle \left\langle \top \;\middle\vert\; \lozenge_{[0,\frac{1}{2}]} E^{B}_{\rm in} \land \square \left(\Phi_B \lor \neg \mathcal{X}^{B}\right) \right\rangle$ |
| $Q$ | $\displaystyle \left\langle \left(\lozenge_{[0,\frac{1}{2}]} E^{B}_{\rm in} \lor \square_{[0,\frac{1}{2}]} \left(\neg E^{B\star}_{\rm in} \lor \neg \mathit{road}^{B\star}_w \mathbin{\mathrm{R}} \neg E^{B\star}_{\rm out}\right)\right) \land \left(\square \left(\Phi_B \lor \neg \mathcal{X}^{B}\right) \lor \square_{[0,\frac{1}{2}]} \left(\neg E^{B\star}_{\rm in} \lor \neg \mathit{road}^{B\star}_w \mathbin{\mathrm{R}} \neg E^{B\star}_{\rm out}\right)\right) \;\middle\vert\; \lozenge_{[0,\frac{1}{2}]} \left(E^{B\star}_{\rm in} \land \mathit{road}^{B\star}_w \mathbin{\mathrm{U}} E^{B\star}_{\rm out}\right) \right\rangle$ |
| $C_{\rm cmt}$ (A) | $\displaystyle \left\langle \top \;\middle\vert\; H^A_2 \land \lozenge_{[0,\frac{1}{2}]} \left(E^{A\star}_{\rm in} \land \mathit{road}^{A\star}_e \mathbin{\mathrm{U}} E^{A\star}_{\rm out}\right) \right\rangle$ |
| $C_{\rm cmt}^{+}$ | $\displaystyle \left\langle \top \;\middle\vert\; H^A_2 \land \lozenge_{[0,\frac{1}{2}]} \left(E^{A\star}_{\rm in} \land \mathit{road}^{A\star}_e \mathbin{\mathrm{U}} E^{A\star}_{\rm out}\right) \land \lozenge_{[0,\frac{1}{2}]} \left(E^{B\star}_{\rm in} \land \mathit{road}^{B\star}_w \mathbin{\mathrm{U}} E^{B\star}_{\rm out}\right) \right\rangle$ |
| $\varphi_B$ | $\left(\mathit{road}_w \land \neg \mathcal{D}_B\right) \mathbin{\mathrm{U}_{[0,10]}} \mathit{exit}_w$ |
| $E_{\rm in}\times J\subseteq\mathrm{sat}^\exists(\varphi_B)$ [numerical] | True |

| Candidate | Exit window $K$ [s] | Reserved cells | $E_{\rm in}\times J\subseteq\Phi_\kappa$ [numerical] | $K_\kappa\preceq Q$ | $C_{\rm cmt}\wedge K_\kappa\preceq C_{\rm cmt}^{+}$ | $C_{\rm req}\parallel(C_{\rm cmt}\wedge K_\kappa)\preceq C_{\rm cmt}\wedge C_{\rm trg}$ |
| --- | --- | --- | --- | --- | --- | --- |
| κ1 | (5.25, 6.25) | 563 | False | True | True | True |
| κ2 | (6, 7.25) | 1233 | True | True | True | True |
| κ3 | (7, 8.25) | 1889 | True | True | True | True |

### B: bargaining and commitment

In [16]:
started = perf_counter()
negotiation = negotiate(B_spec, offers)
result = commit(admission, offers, negotiation, admission.commitment, ledger)
elapsed += perf_counter() - started
table(['Round', 'Proposer', 'Targets (owner, client)', 'Offer', 'Price', 'Utilities (owner, client)', 'Accepted'],
      [(r.number, r.proposer, decimal(tuple(v for _, v in r.targets)), f'κ{r.proposal.offer.index}',
        decimal(r.proposal.price), decimal(tuple(v for _, v in r.utilities)), r.accepted)
       for r in negotiation.history])
table(['Commitment', 'Value'], [('Status', result.status), (r'$C_{\rm cmt}$ after B', result.commitment)], SYMBOLS)
assert result.status == 'COMMITTED'
attempts.append(dict(request=B_spec['request'], admission=admission, offers=offers,
                     negotiation=negotiation, result=result, tried=tuple(tried), elapsed=elapsed))
ledger = result.ledger

| Round | Proposer | Targets (owner, client) | Offer | Price | Utilities (owner, client) | Accepted |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | owner | (4.0, 4.0) | κ2 | 9.0 | (4.0, 1.0) | False |
| 2 | client | (2.5, 2.5) | κ2 | 7.5 | (2.5, 2.5) | True |

| Commitment | Value |
| --- | --- |
| Status | COMMITTED |
| $C_{\rm cmt}$ after B | $\displaystyle \left\langle \top \;\middle\vert\; H^A_2 \land H^B_2 \land \lozenge_{[0,\frac{1}{2}]} \left(E^{A\star}_{\rm in} \land \mathit{road}^{A\star}_e \mathbin{\mathrm{U}} E^{A\star}_{\rm out}\right) \land \lozenge_{[0,\frac{1}{2}]} \left(E^{B\star}_{\rm in} \land \mathit{road}^{B\star}_w \mathbin{\mathrm{U}} E^{B\star}_{\rm out}\right) \right\rangle$ |

### C–I and summary

In [17]:
for spec in REQUESTS[2:]:
    record = attempt(spec, ledger)
    attempts.append(record)
    ledger = record['result'].ledger
table(['Client', 'Services tried: $J$ start [s]', 'Offers', 'Rounds', 'Outcome', 'Exit window $K$ [s]', 'Price',
       'Time [s] (incl. compute)'],
      [(a['request'].client, '; '.join(f'{decimal(entry[0])}: {fate}' for entry, fate in a['tried']),
        len(a['offers']), len(a['negotiation'].history), a['result'].status,
        '—' if a['result'].agreement is None else decimal(a['result'].agreement.offer.exit),
        '—' if a['result'].agreement is None else decimal(a['result'].agreement.price),
        decimal(a['elapsed'])) for a in attempts])
assert [a['result'].status for a in attempts] == ['COMMITTED']*4 + ['NO-BLA'] + ['COMMITTED']*2 + ['NO-BLA'] + ['COMMITTED']*2
assert [b.offer.service.client for b in ledger] == ['A', 'B', 'C', 'D', 'F', 'G', 'H', 'I']
assert [fate for _, fate in attempts[2]['tried']] == ['no certified offer', 'admitted']  # C
assert attempts[4]['admission'] is not None and attempts[4]['offers']  # E: offers, but no agreement on price
assert attempts[7]['admission'] is None and [fate for _, fate in attempts[7]['tried']] == ['entry condition fails']*2  # E again

Running for 40 iterations:   0%|          | 0/40 [00:00<?, ?it/s]

Running for 40 iterations:  10%|█         | 4/40 [00:00<00:01, 23.08it/s]

Running for 40 iterations:  20%|██        | 8/40 [00:00<00:01, 23.75it/s]

Running for 40 iterations:  30%|███       | 12/40 [00:00<00:01, 23.91it/s]

Running for 40 iterations:  40%|████      | 16/40 [00:00<00:01, 23.54it/s]

Running for 40 iterations:  50%|█████     | 20/40 [00:00<00:00, 23.29it/s]

... (111 outputs left out)

Running for 40 iterations:  90%|█████████ | 36/40 [00:01<00:00, 23.77it/s]

Running for 40 iterations: 100%|██████████| 40/40 [00:01<00:00, 25.90it/s]

Running for 40 iterations: 100%|██████████| 40/40 [00:01<00:00, 24.67it/s]

| Client | Services tried: $J$ start [s] | Offers | Rounds | Outcome | Exit window $K$ [s] | Price | Time [s] (incl. compute) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| A | 0.0: admitted | 2 | 2 | COMMITTED | (6.0, 7.2) | 7.5 | 14.5 |
| B | 0.0: admitted | 2 | 2 | COMMITTED | (6.0, 7.2) | 7.5 | 11.3 |
| C | 0.0: no certified offer; 5.0: admitted | 2 | 2 | COMMITTED | (11.0, 12.2) | 7.5 | 19.0 |
| D | 5.0: admitted | 2 | 2 | COMMITTED | (9.0, 10.0) | 7.5 | 11.9 |
| E | 6.0: admitted | 2 | 3 | NO-BLA | — | — | 11.4 |
| F | 8.0: admitted | 3 | 3 | COMMITTED | (11.2, 12.2) | 9.8 | 10.7 |
| G | 8.0: admitted | 3 | 3 | COMMITTED | (11.2, 12.0) | 9.8 | 11.6 |
| E | 8.0: entry condition fails; 9.0: entry condition fails | 0 | 0 | NO-BLA | — | — | 3.9 |
| H | 10.0: admitted | 3 | 3 | COMMITTED | (16.0, 17.5) | 9.8 | 13.6 |
| I | 10.0: admitted | 2 | 2 | COMMITTED | (17.2, 18.8) | 7.5 | 12.2 |

In [18]:
# Committed corridors, projected over (t, h, v) onto the roads (grey); one solid colour per client.
COLORS = dict(zip([b.offer.service.client for b in ledger],
                  ('royalblue', 'crimson', 'seagreen', 'purple', 'darkorange', 'teal', 'magenta', 'goldenrod')))
SOLID = {k: [[0, c], [1, c]] for k, c in COLORS.items()}  # plotly colour scale of one colour
figure = impl.plot(*((ROAD[d](impl), dict(colorscale='Greys', value=.3)) for d in MAP),
                   *((b.offer.corridor, dict(colorscale=SOLID[b.offer.service.client], value=.7)) for b in ledger),
                   method='bitmap', axes=('x', 'y'),
                   layout=dict(title='Committed corridors: ' + ', '.join(f'{k} ({c})' for k, c in COLORS.items()),
                               width=560, height=540))
figure.show()

In [19]:
# The same corridors over (x, y, t), projected over (h, v); each on its own absolute times [a, a+9].
figure = impl.plot(*((b.offer.corridor, dict(colorscale=SOLID[b.offer.service.client], opacity=.6,
                                              min_bounds=[x[0], y[0], b.offer.service.entry[0]],
                                              max_bounds=[x[-1], y[-1], b.offer.service.entry[0] + HORIZON]))
                     for b in ledger),
                   method='isosurface', axes=('x', 'y', 't'),
                   layout=dict(title='Committed corridors over time', width=700, height=800))
# Each isosurface sets the scene to its own bounds; span every corridor's times instead.
figure.update_layout(scene=dict(zaxis_range=[0, max(b.offer.service.entry[0] for b in ledger) + HORIZON],
                                aspectmode='manual', aspectratio=dict(x=1, y=1, z=1.5)))
figure.show()

## Reproduction

```sh
uv run --group notebooks python scripts/run_ltms_notebook.py --save
```